# Étape 2 - Partie 2 : Moteur de Recherche (Stratégie Question-Réponse)
**Objectif :** Pour chaque question du jeu de test (`test_unique`), trouver les $k=10$ réponses les plus pertinentes dans la base de connaissances (`train_unique`) en utilisant la similarité cosinus.
**Méthodes :** TF-IDF et Word2Vec.

In [ ]:
# installer : !pip install nltk sentence-transformers

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
import string

# Chargement automatique des fichiers
try:
    df_train = pd.read_csv('train_unique_e2p2.csv')
    df_test = pd.read_csv('test_unique_e2p2.csv')
except:
    df_train = pd.read_csv('data/train_unique_e2p2.csv')
    df_test = pd.read_csv('data/test_unique_e2p2.csv')

print(f"Données prêtes : {len(df_train)} réponses en base et {len(df_test)} questions de test.")

In [ ]:
# ==========================================
# FONCTION DE NETTOYAGE COMMUNE AU GROUPE
# ==========================================
stop_words = set(stopwords.words('english'))

def text_process(mess):
    lower_mess = mess.lower()
    # Suppression de la ponctuation
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    # Suppression des stop words et séparation en mots
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

## Méthode 1 : Vectorisation par TF-IDF
Le TF-IDF donne un poids aux mots. On vectorise les réponses de l'entraînement pour créer notre "base de recherche", puis on vectorise les questions du test dans le même espace mathématique pour calculer la distance.

In [ ]:
k = 10 
# Utilisation de l'analyseur d'Idir
vectorizer = TfidfVectorizer(analyzer=text_process)

# Transformation en matrices mathématiques
X_train = vectorizer.fit_transform(df_train['Response'])
X_test = vectorizer.transform(df_test['Context'])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : TF-IDF ---\n")

for i in range(len(df_test)):
    similarites = cosine_similarity(X_test[i], X_train).flatten()
    indices_top_k = similarites.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : {df_test.iloc[i]['Context'][:120]}...")
    
    # Affichage de la meilleure réponse
    best_idx = indices_top_k[0]
    print(f"MEILLEURE RÉPONSE (Score: {similarites[best_idx]:.4f}) :")
    print(f"   \"{df_train.iloc[best_idx]['Response'][:300]}...\"")
    
    print(f"Liste des {k} indices : {indices_top_k.tolist()}")
    print("-" * 80)

## Méthode 2 : Vectorisation Sémantique par Word2Vec
Ici, l'IA essaie de comprendre le "sens" des phrases. On transforme chaque mot en coordonnées spatiales, puis on fait la moyenne de ces coordonnées pour obtenir le vecteur global de la phrase.

In [ ]:
# Préparation des phrases avec le nettoyage
phrases_train = df_train['Response'].apply(text_process).tolist()
phrases_test = df_test['Context'].apply(text_process).tolist()

# Entraînement du modèle Word2Vec
w2v_model = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1)

# Fonction pour obtenir le vecteur moyen d'une phrase
def get_sentence_vector(words, model):
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

vect_train = np.array([get_sentence_vector(p, w2v_model) for p in phrases_train])
vect_test = np.array([get_sentence_vector(p, w2v_model) for p in phrases_test])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : WORD2VEC ---\n")

for i in range(len(df_test)):
    sims = cosine_similarity(vect_test[i].reshape(1, -1), vect_train).flatten()
    indices = sims.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : {df_test.iloc[i]['Context'][:120]}...")
    print(f"MEILLEURE RÉPONSE (Score: {sims[indices[0]]:.4f}) :")
    print(f"   \"{df_train.iloc[indices[0]]['Response'][:300]}...\"")
    
    print(f"Liste des {k} indices : {indices.tolist()}")
    print("-" * 80)

## Méthode 3 : Vectorisation Contextuelle par BERT (Sentence-BERT)
BERT est un modèle d'Intelligence Artificielle de type "Transformer". Contrairement à Word2Vec qui regarde les mots un par un, BERT lit la phrase entière dans les deux sens (bidirectionnel) pour comprendre le contexte exact de chaque mot avant de générer le vecteur mathématique.

In [ ]:
# On utilise Sentence-BERT pour le clustering de phrases
model_bert = SentenceTransformer('all-MiniLM-L6-v2')

# Encodage (BERT travaille mieux sur le texte brut ou légèrement nettoyé)
# On utilise 'Response_clean' si disponible ou juste le texte
print("Encodage BERT en cours...")
bert_train = model_bert.encode(df_train['Response'].tolist())
bert_test = model_bert.encode(df_test['Context'].tolist())

print("\n--- RÉSULTATS DU MOTEUR DE RECHERCHE : BERT ---\n")

for i in range(len(df_test)):
    sims_bert = cosine_similarity(bert_test[i].reshape(1, -1), bert_train).flatten()
    indices_bert = sims_bert.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : {df_test.iloc[i]['Context'][:120]}...")
    print(f"MEILLEURE RÉPONSE (Score: {sims_bert[indices_bert[0]]:.4f}) :")
    print(f"   \"{df_train.iloc[indices_bert[0]]['Response'][:300]}...\"")
    
    print(f"Liste des {k} indices : {indices_bert.tolist()}")
    print("-" * 80)